# Data Collection

## NYT

In [ ]:
"""
nyt_longevity_scraper.py
------------------------
NYT Archive API pull (2010‑present) filtered for longevity / healthy‑aging
keywords.   Requires: python-dotenv, requests, pandas, tqdm.
"""
import os, re, time, datetime as dt, requests, pandas as pd
from dotenv import load_dotenv                      
from tqdm import tqdm                                 

# ── 0.  CONFIG ────────────────────────────────────────────────────────────────
load_dotenv() 
API_KEY = os.getenv("NYT_API_KEY") 
if not API_KEY:
    raise RuntimeError("Set NYT_API_KEY in your .env file")

BASE  = "https://api.nytimes.com/svc"
OUT_DIR = "nyt_monthly_csv"                        
os.makedirs(OUT_DIR, exist_ok=True)

# keyword regex (add more if desired)
KEYWORDS = re.compile(
    r"\b(longevity|healthy[ -]ag(?:ing|e?ing)|biohacking|anti[ -]ag(?:ing|e?ing))\b",
    flags=re.I
)

session = requests.Session()
session.params = {"api-key": API_KEY}             


# ── 1.  HELPERS ───────────────────────────────────────────────────────────────
def archive_month(year: int, month: int) -> pd.DataFrame:
    """Fetch one month; return *raw* DataFrame."""
    url = f"{BASE}/archive/v1/{year}/{month}.json"
    r = session.get(url, timeout=60)
    r.raise_for_status()
    docs = r.json()["response"]["docs"]
    return pd.json_normalize(docs)

def filter_keywords(df: pd.DataFrame) -> pd.DataFrame:
    """Keep rows whose text fields match KEYWORDS regex."""
    text = (
        df["abstract"].fillna("") + " " +
        df["lead_paragraph"].fillna("") + " " +
        df["snippet"].fillna("")
    )
    return df[text.str.contains(KEYWORDS, regex=True)]

def month_file(year: int, month: int) -> str:
    """Path for monthly CSV."""
    return f"{OUT_DIR}/nyt_{year}_{month:02d}.csv"


# ── 2.  MAIN LOOP ─────────────────────────────────────────────────────────────
def collect_nyt_aging(start_year: int = 2010):
    today = dt.datetime.today()
    months_total = (today.year - start_year) * 12 + today.month
    progress = tqdm(total=months_total, desc="NYT months")

    for year in range(start_year, today.year + 1):
        max_month = today.month if year == today.year else 12
        for month in range(1, max_month + 1):
            progress.update(1)
            fn = month_file(year, month)
            if os.path.exists(fn):                     # resume support
                continue

            try:
                df_raw  = archive_month(year, month)
                df_keep = filter_keywords(df_raw)

                if not df_keep.empty:
                    df_keep.to_csv(fn, index=False)
            except requests.HTTPError as e:
                print(f"HTTP error {year}-{month:02d}: {e}")
            except Exception as ex:
                print(f"Other error {year}-{month:02d}: {ex}")

            time.sleep(12.5)  # 5 calls/min ⇒ 12 s per call keeps us safe

    progress.close()


# ── 3.  CONCAT AFTER RUN ─────────────────────────────────────────────────────
def concat_all() -> pd.DataFrame:
    frames = [
        pd.read_csv(os.path.join(OUT_DIR, f))
        for f in os.listdir(OUT_DIR)
        if f.endswith(".csv")
    ]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# ── 4.  RUN ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    collect_nyt_aging(start_year=2010)
    df_all = concat_all()
    df_all.to_csv("nyt_longevity_2010‑present.csv", index=False)
    print(f"✔  Saved {len(df_all):,} filtered NYT articles to nyt_longevity_2010‑present.csv")


## Guardian API

Manual for using API:
https://open-platform.theguardian.com/documentation/

In [1]:
import os, time, datetime as dt, requests, pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from dotenv import load_dotenv                       

load_dotenv() 
API_KEY = os.getenv("GUARDIAN_API_KEY") 
BASE_URL = "https://content.guardianapis.com/search"

# constant values
FIELDS = "bodyText,headline,trailText,byline,wordcount"
TAGS   = "keyword,contributor"
BLOCKS = "body"  

PAGE_SIZE = 200       
SLEEP_SEC = 1.1

In [2]:
def build_query(key, core_keys):
    parts=[]         # singular
    if not key.endswith("s"):
        parts.append('"' + key + "s" + '"')    # make it plural
    parts.append('"' + key + '"')  # append singular
    # parts.append(key + "*")        # wildcard (myokine*)

    seen, out = set(), []
    for p in parts:
        if p not in seen:
            seen.add(p); out.append(p)
    
    new_out = []
    for i in range(len(out)):
        for k in core_keys:
            new_out.append(out[i] + " AND " + '"' + k + '"')
    
    new_out = " OR ".join(new_out)
    
    return new_out


def fetch_page(page: int, *, query: str, from_date: str, to_date: str,
               page_size: int = PAGE_SIZE,
               query_fields: str | None = None,
               order_by: str | None = None,
               section: str | None = None,
               tag: str | None = None,
               content_type: str | None = None) -> dict:
    params = {
        "q": query,
        "from-date": from_date,
        "to-date": to_date,
        "page-size": page_size,
        "page": page,
        "show-fields": FIELDS,
        "show-tags": TAGS,
        "show-blocks": BLOCKS,
        "use-date": "published",
        "api-key": API_KEY,
    }
    if query_fields:
        params["query-fields"] = query_fields       # <— important
    if order_by:
        params["order-by"] = order_by
    if section:
        params["section"] = section
    if content_type:
        params["type"] = content_type
    if tag:          
        params["tag"] = tag 
    
    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()

    # debug_url = requests.Request("GET", BASE_URL, params=params).prepare().url
    # print("URL:", debug_url)
    
    return r.json()["response"]

def extract_links(html: str) -> list[str]:
    """Return all href links from an HTML fragment."""
    soup = BeautifulSoup(html or "", "html.parser")
    return [a["href"] for a in soup.find_all("a", href=True)]

def run(KEYS: list[str],
        FROM: str,
        TO: str,
        CORE_KEYS:list[str] = ["aging","ageing","healthy aging","healthy ageing","longevity","anti-ageing","anti-aging","anti ageing","anti aging","biohacking"],
        PATH: str | None = None,
        FILENAME: str | None = None,
        max_pages: int | None = None, #the maximum number of results pages to fetch from the api
        max_articles: int | None = None, # the maximum total number of articles we want
        page_size: int = PAGE_SIZE, #page_size = articles per page
        query_fields: str | None = None,
        order_by: str | None = None,
        section: str | None = None,
        tag: str | None = None,
        content_type: str | None = None):
    # concatenating the query
    QUERY = build_query(KEYS, CORE_KEYS)
    print("QUERY:", QUERY)
    
    first = fetch_page(1, query=QUERY, from_date=FROM, to_date=TO, page_size=page_size, query_fields = query_fields, order_by=order_by, section=section,tag=tag, content_type=content_type)
    total_results = first["total"]
    print(f"{KEYS}: Total matching articles found = {total_results:,}")
    total_pages = first["pages"]
    pages = min(total_pages, max_pages) if max_pages else total_pages
    results = list(first["results"])

    pbar = tqdm(range(2, pages + 1), desc="Guardian pages")
    for p in pbar:
        time.sleep(SLEEP_SEC)
        resp = fetch_page(p, query=QUERY, from_date=FROM, to_date=TO, page_size=page_size, query_fields = query_fields, order_by=order_by, section=section,tag=tag, content_type=content_type)
        results.extend(resp["results"])
        #in case the length of results exceeds the maximum number of articles needed -> cut it
        if max_articles and len(results) >= max_articles:
            results = results[:max_articles]
            break

    # Flatten into a DataFrame
    rows = []
    for item in results:
        fields = item.get("fields", {})
        blocks = item.get("blocks", {})
        
        body_text = fields.get("bodyText", "")
        body_html = fields.get("bodyHtml", "")
        
        body_blocks = blocks.get("body", [])   # list of block dicts
        
        # blocks fallback
        if not body_html and body_blocks:
            body_html = "\n".join(b.get("bodyHtml", "") for b in body_blocks)
        if not body_text and body_blocks:
            body_text = "\n\n".join(b.get("bodyTextSummary", "") for b in body_blocks)
        
        links = extract_links(body_html)

        # authors from contributor tags
        authors = [t["webTitle"] for t in item.get("tags", []) if t.get("type") == "contributor"]
        keywords = [t["webTitle"] for t in item.get("tags", []) if t.get("type") == "keyword"]

        rows.append({
            "id": item.get("id"),
            "section": item.get("sectionName"),
            "pub_date": item.get("webPublicationDate"),
            "web_url": item.get("webUrl"),
            "headline": fields.get("headline"),
            "trailText": fields.get("trailText"),
            "byline": fields.get("byline"),
            "wordcount": fields.get("wordcount"),
            "bodyText": body_text,
            "links": "; ".join(links),
            "authors": "; ".join(authors),
            "tags": "; ".join(keywords),
        })

    df = pd.DataFrame(rows)

    return df

## First Fetching

# Second Fetching

In [7]:
KEYWORDS = [
     'cell',
     'mice',
     'telomeres',
     'senescent',
     'senescent cells',
     'ageing process',
     'stem',
     'dna',
     'biological age',
     'plasma',
     'gene',
     'young blood',
     'treatments',
     'anti ageing',
     'immune', 
     'coray',
     'wyss coray',
     'wyss',
     'stem cells',
     'tissues',
     'mouse',
     'protein',
     'experiments',
     'tropical',
     'related diseases',
     'tissue',
     'villeda',
     'protein',
     'meditation'
]

PATH = "../GuardianAPI/"
FILENAME = "guardian_data_2010_2025_topic0_v2.csv"

all_frames = []

core_keys = ["aging","ageing",
             "healthy aging","healthy ageing",
             "longevity",
             "anti-ageing","anti-aging","anti ageing","anti aging",
             "biohacking", "living longer", 
             "well ageing", "well aging", "ageing well", "aging well",
             "slow ageing", "slow aging"]

for key in KEYWORDS:
    year_frames = []
    for year in range(2010,2025+1):
        FROM=f"{year}-01-01"
        TO = f"{year}-12-31"

        df = run(key, FROM, TO, CORE_KEYS = core_keys, query_fields="headline,body,trailText", max_pages=25, max_articles=None, 
                 page_size=100, order_by="relevance",tag="type/article", content_type="article")
        
        if not df.empty:
                df = df.copy()
                df["topic_key"] = key
                year_frames.append(df)
    
    if year_frames:
        topic_df = pd.concat(year_frames, ignore_index=True, sort=False)
        all_frames.append(topic_df)

combined = pd.concat(all_frames, ignore_index=True, sort=False) if all_frames else pd.DataFrame()
combined.drop_duplicates(subset=["id"], inplace=True)

os.makedirs(PATH, exist_ok=True)
out_file = os.path.join(PATH, FILENAME)
combined.to_csv(out_file, index=False)
print(f"Saved {len(combined):,} rows to {out_file}")

QUERY: "cells" AND "aging" OR "cells" AND "ageing" OR "cells" AND "healthy aging" OR "cells" AND "healthy ageing" OR "cells" AND "longevity" OR "cells" AND "anti-ageing" OR "cells" AND "anti-aging" OR "cells" AND "anti ageing" OR "cells" AND "anti aging" OR "cells" AND "biohacking" OR "cells" AND "living longer" OR "cells" AND "well ageing" OR "cells" AND "well aging" OR "cells" AND "ageing well" OR "cells" AND "aging well" OR "cells" AND "slow ageing" OR "cells" AND "slow aging" OR "cell" AND "aging" OR "cell" AND "ageing" OR "cell" AND "healthy aging" OR "cell" AND "healthy ageing" OR "cell" AND "longevity" OR "cell" AND "anti-ageing" OR "cell" AND "anti-aging" OR "cell" AND "anti ageing" OR "cell" AND "anti aging" OR "cell" AND "biohacking" OR "cell" AND "living longer" OR "cell" AND "well ageing" OR "cell" AND "well aging" OR "cell" AND "ageing well" OR "cell" AND "aging well" OR "cell" AND "slow ageing" OR "cell" AND "slow aging"
cell: Total matching articles found = 35


Guardian pages: 0it [00:00, ?it/s]

QUERY: "cells" AND "aging" OR "cells" AND "ageing" OR "cells" AND "healthy aging" OR "cells" AND "healthy ageing" OR "cells" AND "longevity" OR "cells" AND "anti-ageing" OR "cells" AND "anti-aging" OR "cells" AND "anti ageing" OR "cells" AND "anti aging" OR "cells" AND "biohacking" OR "cells" AND "living longer" OR "cells" AND "well ageing" OR "cells" AND "well aging" OR "cells" AND "ageing well" OR "cells" AND "aging well" OR "cells" AND "slow ageing" OR "cells" AND "slow aging" OR "cell" AND "aging" OR "cell" AND "ageing" OR "cell" AND "healthy aging" OR "cell" AND "healthy ageing" OR "cell" AND "longevity" OR "cell" AND "anti-ageing" OR "cell" AND "anti-aging" OR "cell" AND "anti ageing" OR "cell" AND "anti aging" OR "cell" AND "biohacking" OR "cell" AND "living longer" OR "cell" AND "well ageing" OR "cell" AND "well aging" OR "cell" AND "ageing well" OR "cell" AND "aging well" OR "cell" AND "slow ageing" OR "cell" AND "slow aging"


cell: Total matching articles found = 46


Guardian pages: 0it [00:00, ?it/s]

...

Guardian pages: 0it [00:00, ?it/s]

QUERY: "meditations" AND "aging" OR "meditations" AND "ageing" OR "meditations" AND "healthy aging" OR "meditations" AND "healthy ageing" OR "meditations" AND "longevity" OR "meditations" AND "anti-ageing" OR "meditations" AND "anti-aging" OR "meditations" AND "anti ageing" OR "meditations" AND "anti aging" OR "meditations" AND "biohacking" OR "meditations" AND "living longer" OR "meditations" AND "well ageing" OR "meditations" AND "well aging" OR "meditations" AND "ageing well" OR "meditations" AND "aging well" OR "meditations" AND "slow ageing" OR "meditations" AND "slow aging" OR "meditation" AND "aging" OR "meditation" AND "ageing" OR "meditation" AND "healthy aging" OR "meditation" AND "healthy ageing" OR "meditation" AND "longevity" OR "meditation" AND "anti-ageing" OR "meditation" AND "anti-aging" OR "meditation" AND "anti ageing" OR "meditation" AND "anti aging" OR "meditation" AND "biohacking" OR "meditation" AND "living longer" OR "meditation" AND "well ageing" OR "meditation

meditation: Total matching articles found = 19


Guardian pages: 0it [00:00, ?it/s]


Saved 4,451 rows to ../GuardianAPI/guardian_data_2010_2025_topic0_v2.csv
